# Comprehensive Transformer Architecture Implementation - Interview Practice

## Overview
This notebook covers the complete implementation of transformer architectures from scratch, including:
- Core attention mechanisms (single-head, multi-head, MQA, GQA)
- Positional encodings (sinusoidal, RoPE)
- Normalization techniques (LayerNorm, RMSNorm)
- Advanced features (KV Cache, MoE, Quantization)
- Encoder, Decoder, and Encoder-Decoder architectures

**Dataset**: We'll build a character-level language model on a small text corpus.

**Goal**: Master every component of modern transformer architectures for technical interviews.

## Part 1: Setup and Data Preparation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple
from torch.utils.data import Dataset, DataLoader

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [11]:
# TODO: Load a simple text dataset (e.g., tiny shakespeare or create synthetic data)
# TODO: Create character-level vocabulary
# Constants
SEQ_LEN = 32
VOCAB_SIZE = 1000
BATCH_SIZE = 64
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 200

class RandomizedDataset(Dataset):
    def __init__(self, seq_len, vocab_size, num_samples):
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x = torch.randint(0, self.vocab_size, (self.seq_len, ))
        y = torch.randint(0, self.vocab_size, (self.seq_len, ))
        return x, y

# TODO: Implement data preprocessing and create train/val splits
train_data = RandomizedDataset(SEQ_LEN, VOCAB_SIZE, TRAIN_SAMPLES)
val_data = RandomizedDataset(SEQ_LEN, VOCAB_SIZE, VAL_SAMPLES)
        
# TODO: Create DataLoader
train_dataloader = DataLoader(train_data, batch_size = BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size = BATCH_SIZE, shuffle=True)

for x, y in train_dataloader:
    print(f"Train x {x}")
    print(f"Train y {y}")
    break



Train x tensor([[988, 297, 790,  ..., 268, 892, 985],
        [813, 659, 244,  ..., 894, 506, 994],
        [946, 514, 583,  ..., 434, 362, 924],
        ...,
        [812, 297, 578,  ..., 459, 383, 973],
        [471, 518, 216,  ..., 917, 730,  74],
        [768, 630, 316,  ..., 839, 186, 133]])
Train y tensor([[753, 762,  16,  ..., 169, 659,  16],
        [ 30, 489, 843,  ..., 217, 420, 917],
        [564, 609, 411,  ..., 994, 862, 895],
        ...,
        [654,  38, 689,  ..., 822, 212, 546],
        [149, 209, 473,  ..., 227, 768, 972],
        [803, 764,  68,  ...,  49, 126, 226]])


## Part 2: Single Head Attention

Implement the basic scaled dot-product attention mechanism:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # compute attenion
        attn = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_model)

        # apply mask
        if mask:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn, dim=-1)
        attn_weights = torch.matmul(attn_weights, V)

        # final output
        output = self.W_o(attn_weights)
        return output
        

## Part 3: Multi-Head Attention

Extend to multiple attention heads operating in parallel.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Assert d_model is divisible by num_heads
        # TODO: Store d_model, num_heads, d_k (d_model // num_heads)
        # TODO: Initialize Q, K, V projections
        # TODO: Initialize output projection and dropout
        # assert...
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = nn.Dropout(dropout)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        # resize and transpose
        # we want (batch_size, num_heads, seq_len, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)

        if mask:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output)
        
        

## Part 4: Attention Masks

Implement different types of masks for attention.

In [ ]:
def create_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """Create mask for padding tokens (1 for valid, 0 for padding)"""
    # TODO: Return mask of shape (batch_size, 1, 1, seq_len)
    mask = (seq != pad_idx).unsqueeze(1).unsqueez(2)
    return mask

def create_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Create causal mask for autoregressive generation (lower triangular)"""
    # TODO: Return lower triangular mask of shape (1, 1, seq_len, seq_len)
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
    return mask

def combine_masks(padding_mask: Optional[torch.Tensor], 
                  causal_mask: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
    """Combine padding and causal masks"""
    # TODO: Combine masks using logical AND
    if padding_mask is None and causal_mask is None:
        return None
    if padding_mask is None:
        return causal_mask
    if causal_mask is None:
        return padding_mask
    return padding_mask & causal_mask


## Part 5: Feed-Forward Network

Position-wise fully connected feed-forward network.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize two linear layers (d_model -> d_ff -> d_model)
        # TODO: Initialize dropout
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Linear -> ReLU/GELU -> Dropout -> Linear
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        return self.linear2(x)
        

## Part 6: Normalization Layers

### 6.1 Layer Normalization

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # TODO: Initialize learnable parameters (gamma, beta)
        # TODO: Store eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = torch.mean(x, dim=-1)
        std = torch.std(x, dim=-1)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

### 6.2 RMS Normalization

Used in LLaMA and other modern models.

In [13]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # TODO: Initialize weight parameter
        # TODO: Store eps
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute RMS: sqrt(mean(x^2) + eps)
        # TODO: Normalize and scale
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        return self.weight * x / rms
        

## Part 7: Positional Encodings

### 7.1 Sinusoidal Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # (d_model, max_len)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float)
        div_term = 1.0 / (10000 ** torch.arange(0, d_model, 2, dtype=torch.float))
        pe[:, 0::2] = torch.cos(position.unsqueeze(1) * div_term)
        pe[:, 1::2] = torch.sin(position.unsqueeze(1) * div_term)
        pe = pe.unsqueeze(0) # [1, L, D]
        self.register_buffer('pe', pe)
                         
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Add positional encoding to input
        seq_len = x.shape[-2]
        return x + self.pe[:, :seq_len, :]

### 7.2 Rotary Positional Embeddings (RoPE)

Used in models like LLaMA, GPT-NeoX.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # TODO: Compute inverse frequencies
        # TODO: Create rotation matrices
        assert d_model % 2 == 0
        inv_freq = 1.0 / (10000 ** torch.arange(0, d_model, 2, dtype=torch.float))
        self.register_buffer('inv_freq', inv_freq)

        t = torch.arange(0, max_len, dtype=torch.float)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)

        self.register_buffer('cached_cos', emb.cos())
        self.register_buffer('cached_sin', emb.sin())
                
        
    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.shape(-2)
        x1, x2 = x[..., seq_len//2:], x[..., seq_len//2:]
        return torch.cat([-x2, x1])
        
    def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = q.size(-2)

        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        
        q_rot = cos * q + self.rotate_half(q) * sin
        k_rot = cos * k + self.rotate_half(k) * sin

        return q_rot, k_rot
        

## Part 8: Transformer Blocks

### 8.1 Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, 
                 dropout: float = 0.1, use_rms_norm: bool = False):
        super().__init__()
        # TODO: Initialize multi-head attention
        # TODO: Initialize feed-forward network
        # TODO: Initialize normalization layers (LayerNorm or RMSNorm)
        # TODO: Initialize dropout layers
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        NormLayer = RMSNorm if use_rms_norm else LayerNorm
        self.norm1 = NormLayer(d_model)
        self.norm2 = NormLayer(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Self-attention with pre-norm and residual
        # TODO: Feed-forward with pre-norm and residual
        attn_output = self.attention(self.norm1(x), mask)
        attn_output = x + self.dropout1(attn_output)

        # ffn
        ff_output = self.feed_forward(self.norm2(x))
        ff_output = x + self.norm2(ff_output)
        return ff_output
        

### 8.2 Decoder Block with Cross-Attention

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize masked self-attention
        # TODO: Initialize cross-attention
        # TODO: Initialize feed-forward network
        # TODO: Initialize 3 normalization layers
        # TODO: Initialize dropout layers
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Masked self-attention
        # TODO: Cross-attention (Q from decoder, K,V from encoder)
        # TODO: Feed-forward
        self_attn = self.self_attention(self.norm1(x), src_mask)
        x = x + self.dropout1(self_attn)

        cross_attn = self.cross_attention(self.norm2(x), tgt_mask)
        x = x + self.dropout2(cross_attn)

        ff_output = self.feed_forward(self.norm3(x))
        x = x + self.dropout3(ff_output)

## Part 9: Efficient Attention Variants

### 9.1 Multi-Query Attention (MQA)

Single K,V head shared across all Q heads.

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Multiple Q heads, single K and V
        # TODO: Initialize projections
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_model = d_model

        # Define the proejction matrices
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_k)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Project Q with multiple heads
        # TODO: Project K, V with single head, broadcast to all Q heads
        # TODO: Compute attention
        batch_size, seq_len, _ = x.size()
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2).float()

        # project K, V
        K = self.W_k(x).unsqueeze(1).expand(-1, self.num_heads, -1, -1)
        V = self.W_v(x).unsqueeze(1).expand(-1, self.num_heads, -1, -1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # attn_output
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.num_heads, self.d_k)

        return self.W_o(attn_output)
        

### 9.2 Grouped-Query Attention (GQA)

Groups of Q heads share K,V heads (middle ground between MHA and MQA).

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Assert num_heads is divisible by num_kv_heads
        # TODO: Initialize Q projection (num_heads)
        # TODO: Initialize K, V projections (num_kv_heads)
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.num_queries_per_kv_head = d_model // num_kv_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Project Q, K, V
        # TODO: Repeat K, V heads to match Q heads
        # TODO: Compute attention
        # 3rd dimension = num queries per kv head, so the kv heads you need to unsqueeze and broadcast
        batch_size, seq_len, _ = x.size()

        # Q - reshape
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        # Q - grouped
        Q = Q.view(batch_size, self.num_kv_heads, self.num_queries_per_kv_head, seq_len, self.d_k)

        # K - reshape
        K = self.W_k(x).view(batch_size, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2).unsqueeze(2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2).unsqueeze(2)

        # scores
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask:
            scores = scores.masked_fill(scores == 0, float('-inf'))

        # attn weights
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # attn output
        attn_output = torch.matmul(attn_weights, V)
        # (batch_size, seq_len, , d_k)
        attn_output = attn_output.view(batch_size, self.num_kv_heads * self.num_queries_per_kv_head, seq_len, self.d_k).transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output)        

### 9.3 Cross Attention

Key differences from self-attention:

- Two inputs: Takes query_input and kv_input as separate parameters
- Asymmetric computation: Q is computed from one sequence, while K and V come from another
- Different sequence lengths: query_len and kv_len can be different (e.g., decoder attending to encoder)
- Use case: Essential for encoder-decoder architectures (like the original Transformer for translation)

In [ ]:
class CrossAttention(nn.Module):
    """Cross-attention where Q comes from one sequence and K,V from another"""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        
    
    def forward(self, 
                query_input: torch.Tensor, 
                kv_input: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            query_input: (batch, query_seq_len, d_model) - e.g., decoder states
            kv_input: (batch, kv_seq_len, d_model) - e.g., encoder outputs
            mask: Optional attention mask
        Returns:
            output: (batch, query_seq_len, d_model)
        """
        batch_size = query_input.size(0)
        query_len = query_input.size(1)
        kv_len = kv_input.size(1)

        # query 
        Q = self.W_q(query_input).view(batch_size, query_len, self.num_heads, self.d_k).transpose(1,2)
        K = self.W_k(kv_input).view(batch_size, kv_len, self.num_heads, self.d_k).transpose(1,2)
        V = self.W_v(kv_input).view(batch_size, kv_len, self.num_heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask:
            scores = scores.masked_fill(scores == 0, float('-inf'))

        # attn_weights
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # attn_output
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1,2).view(batch_size, query_len, self.num_heads, self.d_k).contiguous().view(batch_size, query_len, self.d_model)

        return self.W_o(attn_output)
        

## Part 10: KV Cache

Cache key-value pairs for efficient autoregressive generation.

In [ ]:
class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def _shape_qkv(self, x: torch.Tensor) -> torch.Tensor:
        """[B, L, D] -> [B, H, L, d_k]"""
        B, L, _ = x.shape
        x = x.view(B, L, self.num_heads, self.d_k).transpose(1,2)
        return x.contiguous()

    @staticmethod
    def _make_causal_mask(B: int, Lq:int, Lk: int, device: torch.device) -> torch.Tensor:
        """[1,1,Lq,Lk] where with True where allowed, Flase where masked """
        i = torch.arange(Lq, device=device)[:, None]
        j = torch.arange(Lk, device = device)[None, :]
        mask = (j <= i + (Lk - Lq))
        return mask.view(1, 1, Lq, Lk)

    def forward(
        self,
        x: torch.Tensor, # [B, Lq, D]
        cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        attn_mask = Optional[torch.Tensor] = None,
        is_causal: bool = True,
        need_weights: bool = False,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor], Optional[torch.Tensor]]:
        B, Lq, _ = x.shape
        device = x.device

        # Project Q, K, V
        Q = self._shape_qkv(self.W_q(x))
        K_new = self._shape_qkv(self.W_k(x))
        V_new = self._shape_qkv(self.W_v(x))

        # Append to cache (or start it)
        if cache is None:
            K_cat, V_cat = K_new, V_new # [B, H, Lq, d_k]
        else:
            k_prev, V_prev = cache
            K_cat = torch.cat([K_prev, K_new], dim=2) # time dim = 2
            V_cat = torch.cat([V_pre, V_new], dim=2)

        # length query 
        Lk = K_cat.size(2)

        # handle attention_mask 
        if attn_mask is None and is_causal:
            attn_mask_b = self._make_causal_mask(B, Lq, Lk, device)
        elif attn_mask is not None:
            attn_mask_b = attn_mask
            if addn_mask_b.dtype != torch.bool:
                attn_mask_b = attn_mask_b != 0
        else:
            attn_mask_b = None

        # scaled dot product Q @ K^T -> [B, H, Lq, Lk]
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V_cat)
        context = context.transpose(1,2).contiguous().view(B, Lq, self.d_model)
        out = self.W_o(context)

        return (out, (K_cat, V_cat), attn_weights if need_weights else None)



# Transformer Block (Decoder-style)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__int__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttentionWithCache(d_model, num_heads, dropout)
        ...

    def forward(
        self,
        x: torch.Tensor,
        cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        attn_mask: Optional[torch.Tensor] = None,
        is_causal: bool = True,
        need_weights: bool = False
    ):
        h, new_cache, attn_w = self.attn(
            self.ln1(x),
            cache = cache,
            attn_mask =attn_mask,
            is_causal=is_causal,
            need_weights=need_weights
        )
        x = x + h
        x = x + self.ff(self.ln2(x))
        return x, new_cache, attn_w


class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, d_ff, dropout=dropout) for _ in range(num_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor, # [B, L, D],
        caches: Optional[List[Optional[Tuple[torch.Tensor, torch.Tensor]]]] = None,
        attn_mask = Optional[torch.Tensor] = None,
        is_causal: bool = True,
        need_weights: bool = False
    ):
        if caches is None:
            caches = [None] * len(self.layers)
        new_caches = []
        attn_weights_list = [] if need_weights else None

        for layer, layer_cache in zip(self.layers, caches):
            x, nc, attn_w = layer(
                x, cache=layer_cache, attn_mask=attn_mask, is_causal=is_causal, need_weights=need_weights
            )
            new_caches.append(nc)
            if need_weights:
                attn_weights_list.append(attn_w)

        x = self.ln_f(x)
        return x, new_caches, attn_weights_list


## Part 11: Latent Attention

Compress K, V to lower-dimensional latent space.

In [ ]:
class LatentAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_latents: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize latent queries
        # TODO: Initialize cross-attention to compress input
        # TODO: Initialize regular attention in latent space
        # TODO: Initialize projection back to original space
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Compress input to latent space
        # TODO: Apply attention in latent space
        # TODO: Project back to original dimension
        pass

## Part 12: Quantization

### 12.1 Symmetric Quantization

In [ ]:
def quantize_tensor(tensor: torch.Tensor, num_bits: int = 8) -> Tuple[torch.Tensor, float]:
    """Quantize tensor to int representation"""
    # TODO: Compute scale factor
    # TODO: Quantize to integers
    # TODO: Return quantized tensor and scale
    qmin = ()

def dequantize_tensor(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize back to float"""
    # TODO: Multiply by scale
    pass

class QuantizedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_bits: int = 8):
        super().__init__()
        # TODO: Initialize weight and bias
        # TODO: Quantize weight during initialization
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Dequantize weight
        # TODO: Apply linear transformation
        pass

## Part 13: Mixture of Experts (MoE)

### 13.1 Single-Gate MoE

In [ ]:
class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Create a simple FFN expert
        self.net = FeedForward(d_model, d_ff, dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Apply FFN
        return self.net(x)

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, 
                 top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        # TODO: Create multiple expert networks
        # TODO: Create gating network
        # TODO: Store top_k
        self.num_experts = num_experts
        self.top_k = top_k

        self.experts = nn.ModuleList(
            [Expert(d_model, d_ff, dropout) for _ in range(num_experts)]
        )

        self.gate = nn.Linear(d_model, num_experts)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute gating scores
        # TODO: Select top-k experts per token
        # TODO: Route tokens to selected experts
        # TODO: Combine expert outputs with gating weights
        batch_size, seq_len, d_model = x.size()

        x_flat = x.view(-1, d_model)
        gate_scores = self.gate(x_flat)
        top_k_scores, top_k_indices = torch.topk(gate_scores, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_scors, dim=-1)

        output = torch.zeros_like(x_flat)

        for i in range(self.top_k):
            expert_idx = top_k_indices[:, i]
            expert_weight = top_k_weights[:, i:i+1]

            # group by expert
            for expert_id in range(self.num_experts):
                mask = expert_idx == expert_id
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_output = self.experts[expert_id](expert_input)
                    output[mask] += expert_weight[mask] * expert_output

        return output.view(batch_size, seq_len, d_model)
        

### 13.2 Multi-Gate MoE

Different gates for different tasks/modalities.

In [ ]:
class MultiGateMixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, 
                 num_gates: int = 2, top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        # TODO: Create shared expert networks
        # TODO: Create multiple gating networks
        # TODO: Create gate selection mechanism
        pass
    
    def forward(self, x: torch.Tensor, gate_idx: int = 0) -> torch.Tensor:
        # TODO: Select appropriate gate
        # TODO: Route through experts
        pass

## Part 14: Learning Rate Schedulers

In [ ]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, 
                 min_lr: float = 1e-6, max_lr: float = 1e-3):
        # TODO: Store parameters
        pass
    
    def step(self, current_step: int):
        # TODO: Implement warmup phase (linear increase)
        # TODO: Implement cosine decay phase
        pass

class InverseSqrtScheduler:
    def __init__(self, optimizer, warmup_steps: int, d_model: int):
        # TODO: Implement scheduler from "Attention is All You Need"
        # lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
        pass
    
    def step(self, current_step: int):
        # TODO: Update learning rate
        pass

## Part 15: Complete Model Assembly

### 15.1 Decoder-Only Language Model (GPT-style)

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, 
                 num_layers: int, d_ff: int, max_len: int = 512, 
                 dropout: float = 0.1, use_rope: bool = False,
                 use_moe: bool = False, use_gqa: bool = False):
        super().__init__()
        # TODO: Token embedding
        # TODO: Positional encoding (sinusoidal or RoPE)
        # TODO: Stack of transformer decoder blocks
        # TODO: Output projection to vocabulary
        self.d_model = d_model
        self.use_rope = use_rope

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        if use_rope:
            self.pos_encoding = RotaryPositionalEmbedding(d_model, max_len)
        else:
            self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)

        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerEncoderBlock
        ])
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Embed tokens
        # TODO: Add positional encoding
        # TODO: Apply transformer blocks
        # TODO: Project to vocabulary
        pass
    
    def generate(self, prompt: torch.Tensor, max_new_tokens: int, 
                 temperature: float = 1.0, top_k: Optional[int] = None) -> torch.Tensor:
        # TODO: Implement autoregressive generation
        # TODO: Use KV cache if available
        # TODO: Apply temperature and top-k sampling
        pass

### 15.2 Encoder-Decoder Model (T5-style)

In [ ]:
class TransformerEncoderDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_encoder_layers: int, num_decoder_layers: int,
                 d_ff: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        # TODO: Encoder and decoder embeddings
        # TODO: Positional encodings
        # TODO: Encoder stack
        # TODO: Decoder stack with cross-attention
        # TODO: Output projection
        pass
    
    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Encode source sequence
        pass
    
    def decode(self, tgt: torch.Tensor, encoder_output: torch.Tensor,
               src_mask: Optional[torch.Tensor] = None,
               tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Decode target sequence with cross-attention
        pass
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Full forward pass
        pass

## Part 16: Training Loop

In [ ]:
# TODO: Initialize model with chosen architecture
# TODO: Define loss function (CrossEntropyLoss)
# TODO: Define optimizer (AdamW)
# TODO: Define scheduler
# TODO: Implement training loop with:
#   - Forward pass
#   - Loss computation
#   - Backward pass
#   - Gradient clipping
#   - Optimizer step
#   - Scheduler step
#   - Logging
# TODO: Track perplexity

## Part 17: Evaluation and Generation

In [ ]:
# TODO: Implement evaluation function
# TODO: Compute validation loss and perplexity
# TODO: Implement text generation function
# TODO: Test different sampling strategies (greedy, top-k, nucleus)

## Part 18: Experiments and Comparisons

### Experiment 1: Compare Attention Mechanisms

In [ ]:
# TODO: Train models with MHA, MQA, and GQA
# TODO: Compare training speed and memory usage
# TODO: Compare final performance

### Experiment 2: Compare Positional Encodings

In [ ]:
# TODO: Train with sinusoidal vs RoPE
# TODO: Test on sequences longer than training length
# TODO: Compare generalization

### Experiment 3: Evaluate MoE

In [ ]:
# TODO: Compare dense model vs MoE with same total parameters
# TODO: Analyze expert specialization
# TODO: Measure inference speed

### Experiment 4: Quantization Impact

In [ ]:
# TODO: Quantize trained model to 8-bit and 4-bit
# TODO: Measure performance degradation
# TODO: Measure memory savings

## Part 19: Interview Questions to Prepare

After implementing the above, be ready to answer:

1. **Architecture Questions**:
   - Why do we scale attention scores by sqrt(d_k)?
   - What's the difference between encoder and decoder self-attention?
   - How does cross-attention work in encoder-decoder models?
   - Why use pre-norm vs post-norm in transformer blocks?

2. **Efficiency Questions**:
   - How does MQA reduce memory compared to MHA?
   - Explain KV cache and its benefits
   - What are the tradeoffs of GQA vs MQA vs MHA?
   - How does quantization work and what's the quality-size tradeoff?

3. **Training Questions**:
   - Why use warmup in learning rate scheduling?
   - Explain gradient clipping and why it's important
   - What causes instability in training large transformers?
   - How does dropout help in transformers?

4. **Advanced Topics**:
   - Compare sinusoidal PE vs RoPE - which is better for long sequences?
   - How does MoE achieve better parameter efficiency?
   - What is the load balancing problem in MoE?
   - Explain latent attention and its computational benefits